# BIST harness: A–E DEĞERLENDİRMESİ (Colab GPU + Ollama)
Değerlendirme CPU'da değil GPU'da koşulur (Bilgisayar 1 CPU'da prompt ~27 tok/sn; ayrıntı `docs/harness_design.md` §14.5). Adımlar: (1) GPU + koşu planı, (2) Ollama, (3) paket, (4) modeller, (5) şablon denetimi, (6) A–E koşusu, (7) indir.
**Kesintiye dayanıklı:** sonuçlar oturum başına Drive'a yazılır (`results.jsonl`/`results.csv`); hücre 6'yı aynı ayarlarla yeniden çalıştırmak kaldığı yerden devam eder. İki oturuma bölmek için `MAX_RUNS` kullanın.
Zip: `python -m experiments.colab.make_eval_bundle --out eval_bundle.zip` (repo kodu + snapshot'lar + sorular; GGUF dahil değil).

In [ ]:
# 1) GPU + koşu planı
import subprocess, os
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], capture_output=True, text=True).stdout.upper()
GPU = 'A100' if 'A100' in name else 'L4' if 'L4' in name else 'T4'   # tanınmayan GPU'da T4 (en yavaş varsayım)
print('GPU sınıfı:', GPU)

In [ ]:
# 2) Ollama kurulumu ve sunucu (yalnızca yerel 127.0.0.1)
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request
os.environ.update(OLLAMA_HOST='127.0.0.1:11434', OLLAMA_NUM_PARALLEL='1', OLLAMA_KEEP_ALIVE='60m')
server = subprocess.Popen(['ollama','serve'], stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
print('Ollama hazır')

In [ ]:
# 3) Paket: eval_bundle.zip'i yükleyin (ya da Drive'dan kopyalayın)
from google.colab import drive, files
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/bist/eval_bundle.zip'   # Drive'da yoksa aşağıdaki satırı açın
if not os.path.exists(BUNDLE):
    up = files.upload(); BUNDLE = next(iter(up))
!rm -rf /content/bist && mkdir -p /content/bist && unzip -o -q {BUNDLE} -d /content/bist
%cd /content/bist
!pip install -q 'pydantic>=2' pandas numpy pyarrow
!ls data/snapshots data/training/v2

In [ ]:
# 4) Modeller: temel gemma3:4b + ince ayarlı GGUF (training/ollama/Modelfile)
!ollama pull gemma3:4b
GGUF = '/content/drive/MyDrive/bist/gemma3-bist-q4_k_m.gguf'   # Colab eğitim çıktısı (outputs/v2/gguf/...); Drive'da yoksa yükleyin
if not os.path.exists(GGUF):
    up = files.upload(); GGUF = next(iter(up))
!cp '{GGUF}' training/ollama/gemma3-bist-q4_k_m.gguf
!cd training/ollama && ollama create gemma3-bist -f Modelfile
!ollama list

In [ ]:
# 5) Şablon denetimi (zorunlu): Modelfile şablonu eğitim metniyle AYNI token sayısını vermeli. Yalnızca prompt_eval_count istenir (num_predict=1)
!HARNESS_LLM_TIMEOUT=600 python -m experiments.check_template --model gemma3-bist --records data/training/v2/val.jsonl --n 2

In [ ]:
# 6) Koşu planı (GPU'ya göre) ve A–E koşusu
N = 40            # soru sayısı; plan çıktısındaki 'toplam <= 180 dk için en büyük --n' değerini aşmayın
MAX_RUNS = 0      # 0: hepsi; iki oturuma bölmek için planın önerdiği --max-runs (ör. 195). Aynı hücreyi sonra tekrar çalıştırın.
OUT = '/content/drive/MyDrive/bist_eval/v2'     # Drive: oturum düşse de sonuçlar kalır; AYNI klasör = kaldığı yerden devam
os.environ['HARNESS_LLM_TIMEOUT'] = '900'
!python -m experiments.colab.plan --gpu {GPU} --n {N} --calibrate {OUT}/results.jsonl
extra = f'--max-runs {MAX_RUNS}' if MAX_RUNS else ''
!python -m experiments.eval_runner --questions data/training/v2/test.jsonl --n {N} --models gemma3:4b gemma3-bist --profiles A B C D E --out {OUT} {extra}

Kesilirse (oturum düştü / `MAX_RUNS` doldu): hücre 1–5'i yeniden çalıştırıp hücre 6'yı AYNI `OUT` ile çalıştırın; biten satırlar atlanır. Plan, ilk koşulardan sonra `--calibrate` ile ölçülen sürelerle yenilenir.

In [ ]:
# 7) Sonuçlar: özet ve indirme
!cat {OUT}/summary.json | head -60
!cd {OUT} && zip -q -r /content/eval_results.zip results.csv results.jsonl summary.json
files.download('/content/eval_results.zip')